# Assignment 3
**Name:** Manpreet Singh
**Subgroup:** 3c25
**Roll No:** 102303357

# QUES 1 K-Fold Cross Validation for Multiple Linear Regression (Least Square Error Fit)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

df = pd.read_csv("USA_Housing.csv")  

### a. Divide the dataset into input features (all columns except price) and output variable (price)

In [2]:
X = df.drop(columns=['Price'])
y = df['Price'].values.reshape(-1, 1)

### b. Scale the values of input features.

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### c. Divide input and output features into five folds. 
### d. Run five iterations, in each iteration consider one-fold as test set and remaining four sets as training set. Find the beta (𝛽) matrix, predicted values, and R2_score for each iteration using least square error fit.

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_r2 = -np.inf
best_beta = None
best_pred = None

for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    X_train_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
    X_test_b = np.hstack([np.ones((X_test.shape[0], 1)), X_test])
    
    beta = np.linalg.pinv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train
    y_pred = X_test_b @ beta
    r2 = r2_score(y_test, y_pred)
    
    if r2 > best_r2:
        best_r2 = r2
        best_beta = beta
        best_pred = y_pred

### e. Use the best value of (𝛽) matrix (for which R2_score is maximum), to train the regressor for 70% of data and test the performance for remaining 30% data

In [5]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

X_train_full_b = np.hstack([np.ones((X_train_full.shape[0], 1)), X_train_full])
X_test_full_b = np.hstack([np.ones((X_test_full.shape[0], 1)), X_test_full])

y_pred_full = X_test_full_b @ best_beta
r2_full = r2_score(y_test_full, y_pred_full)

print("Best beta matrix:\n", best_beta)
print("R2 score on 30% test data:", r2_full)

Best beta matrix:
 [[1.23161736e+06]
 [2.30225051e+05]
 [1.63956839e+05]
 [1.21115120e+05]
 [7.83467170e+02]
 [1.50662447e+05]]
R2 score on 30% test data: 0.9147458156636434


# Ques 2  Concept of Validation set for Multiple Linear Regression (Gradient Descent Optimization)
Consider the same dataset of Q1, rather than dividing the dataset into five folds, divide the dataset into training set (56%), validation set (14%), and test set (30%). \
Consider four different values of learning rate i.e. {0.001,0.01,0.1,1}. Compute the values of regression coefficients for each value of learning rate after 1000 iterations. \
For each set of regression coefficients, compute R2_score for validation and test set and find the best value of regression coefficients

In [8]:
X = df.drop(columns=['Price']).values
y = df['Price'].values.reshape(-1, 1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

X_train_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
X_val_b = np.hstack([np.ones((X_val.shape[0], 1)), X_val])
X_test_b = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

learning_rates = [0.001, 0.01, 0.1, 1]
n_iter = 1000
best_r2_val = -np.inf
best_beta = None

for lr in learning_rates:
    beta = np.zeros((X_train_b.shape[1], 1))
    for _ in range(n_iter):
        gradient = X_train_b.T @ (X_train_b @ beta - y_train) / X_train_b.shape[0]
        beta -= lr * gradient
    
    y_val_pred = X_val_b @ beta
    r2_val = r2_score(y_val, y_val_pred)
    print(f"Learning rate {lr}  R2 on validation set: {r2_val:.4f}")
    if r2_val > best_r2_val:
        best_r2_val = r2_val
        best_beta = beta

y_test_pred = X_test_b @ best_beta
r2_test = r2_score(y_test, y_test_pred)

print("Best beta coefficients:\n", best_beta)
print("R2 score on test set:", r2_test)


Learning rate 0.001  R2 on validation set: -0.8125
Learning rate 0.01  R2 on validation set: 0.9098
Learning rate 0.1  R2 on validation set: 0.9098
Learning rate 1  R2 on validation set: 0.9098
Best beta coefficients:
 [[1232562.51254919]
 [ 230048.76664688]
 [ 163686.93503606]
 [ 121406.94107918]
 [   3117.47363933]
 [ 150655.97459714]]
R2 score on test set: 0.9147434800538763


# Q3: Pre-processing and Multiple Linear Regression

### Download the dataset regarding Car Price Prediction from the following link: https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

url = "http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"

### 1 Load the dataset with following column names ["symboling", "normalized_losses", "make", "fuel_type", "aspiration","num_doors", "body_style", "drive_wheels" "engine_location", "wheel_base", "length", "width", "height", "curb_weight", "engine_type", "num_cylinders", "engine_size", "fuel_system", "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"] and replace all ? values with NaN

In [31]:
columns = ["symboling", "normalized_losses", "make", "fuel_type", "aspiration",
           "num_doors", "body_style", "drive_wheels", "engine_location", "wheel_base",
           "length", "width", "height", "curb_weight", "engine_type", "num_cylinders",
           "engine_size", "fuel_system", "bore", "stroke", "compression_ratio",
           "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"]

df = pd.read_csv(url, names=columns, na_values='?')



### 2  Replace all NaN values with central tendency imputation. Drop the rows with NaN values in price column

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

df.dropna(subset=['price'], inplace=True)

### 3 There are 10 columns in the dataset with non-numeric values. Convert these values to numeric values using following scheme:
(i) For “num_doors” and “num_cylinders”: convert words (number names) to figures for e.g., two to 2
(ii) For "body_style", "drive_wheels": use dummy encoding scheme
(iii) For “make”, “aspiration”, “engine_location”,fuel_type: use label encoding scheme
(iv) For fuel_system: replace values containing string pfi to 1 else all values to 0.
(v) For engine_type: replace values containing string ohc to 1 else all values to 0

In [33]:
word_to_num = {'two': 2, 'four': 4, 'three':3, 'five':5, 'six':6, 'eight':8, 'twelve':12}
df['num_doors'] = df['num_doors'].map(word_to_num)
df['num_cylinders'] = df['num_cylinders'].map(word_to_num)


df = pd.get_dummies(df, columns=['body_style', 'drive_wheels'], drop_first=True)


label_cols = ['make', 'aspiration', 'engine_location', 'fuel_type']
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col])


df['fuel_system'] = df['fuel_system'].apply(lambda x: 1 if 'pfi' in x else 0)


df['engine_type'] = df['engine_type'].apply(lambda x: 1 if 'ohc' in x else 0)


### 4 Divide the dataset into input features (all columns except price) and output variable (price). Scale all input features.

In [35]:
X = df.drop(columns=['price']).values
y = df['price'].values.reshape(-1, 1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 5 Train a linear regressor on 70% of data (using inbuilt linear regression function of Python) and test its performance on remaining 30% of data.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
r2_full = r2_score(y_test, y_pred)

print("R2 score without PCA:", r2_full)

R2 score without PCA: 0.7454875004100331


### 6 Reduce the dimensionality of the feature set using inbuilt PCA decomposition and then again train a linear regressor on 70% of reduced data (using inbuilt linear regression function of Python). Does it lead to any performance improvement on test set?

In [37]:
pca = PCA(n_components=min(X_train.shape[1], X_train.shape[0]))
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

lr_pca = LinearRegression()
lr_pca.fit(X_train_pca, y_train)
y_pred_pca = lr_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)

print("R2 score without PCA:", r2_full)
print("R2 score with PCA:", r2_pca)

R2 score without PCA: 0.7454875004100331
R2 score with PCA: 0.7454875004100325
